## Prompt Datasets, Grouped Rollouts, and Relative Advantage Estimation

Now that we know the architecture of GRPO, let’s step through the exact data pipeline and tensor mechanics: how do we ingest prompts, generate grouped rollouts in parallel, and compute relative advantages step by step?

---

## 1. The Prompt Dataset Structure

Unlike SFT, which needs a prompt and a gold response, or DPO, which needs a prompt, a chosen response, and a rejected response, GRPO only requires prompts and their verification metadata.

```json
[
  {
    "prompt": "Write a Python function `is_palindrome(s: str) -> bool` that ignores case and non-alphanumeric characters.",
    "tests": [
      "assert is_palindrome('A man, a plan, a canal: Panama') == True",
      "assert is_palindrome('race a car') == False"
    ],
    "ground_truth": "True"
  }
]
```

### Key point

No reference answer strings are needed for training. The environment or test runner only needs the prompt and the verification criteria such as test cases, reference values, or regex rules.

## 2. Grouped Rollouts: Sampling and Batch Geometry

In standard autoregressive training, your batch dimension is $B$ (batch size). In GRPO, we also introduce a group dimension $G$ (typically $G \in [4, 16]$).

If your batch has $B = 2$ prompts and the group size is $G = 4$, the forward generation pipeline looks like this:

### Prompts

```text
Prompts (B = 2)
├── Prompt 0: "Solve 2x + 5 = 15"
└── Prompt 1: "Write is_palindrome()"
```

Each prompt is duplicated $G = 4$ times.

### Generation batch

```text
Generation Batch (B × G = 8 parallel streams)
├── Prompt 0 -> Sample o_0,0 -> "2x = 10 -> x = 5"
├── Prompt 0 -> Sample o_0,1 -> "x = 15 - 5 -> x = 10"
├── Prompt 0 -> Sample o_0,2 -> "2x = 10 -> x = 5"
├── Prompt 0 -> Sample o_0,3 -> "2x = 20 -> x = 10"
├── Prompt 1 -> Sample o_1,0 -> "def is_palindrome(s): return s == s[::-1]"
├── Prompt 1 -> Sample o_1,1 -> "def is_palindrome(s): clean = ...; return clean == clean[::-1]"
├── Prompt 1 -> Sample o_1,2 -> "def is_palindrome(s): pass"
└── Prompt 1 -> Sample o_1,3 -> "def is_palindrome(s): return True"
```

### Key engineering factor: sampling temperature

To ensure the $G$ generations explore different reasoning trajectories, sampling must be stochastic ($T > 0$), typically with temperature in the range $[0.6, 1.0]$ and top-$p \approx 0.95$.

If $T = 0$ (greedy decoding), all $G$ completions will be identical, producing zero reward variance and zero gradients.

## 3. Step-by-Step Numerical Example: Computing Relative Advantages

Let’s trace a concrete example for Prompt 0 with $G = 4$.

### Step A: raw reward evaluation

Each rollout $o_{0,i}$ is passed to the reward function, for example a math verifier:

$$
r = [1.0,\ 0.0,\ 1.0,\ 0.0]
$$

### Step B: group mean

$$
\mu = \frac{1.0 + 0.0 + 1.0 + 0.0}{4} = \frac{2.0}{4} = 0.5
$$

### Step C: group standard deviation

$$
\sigma = \sqrt{\frac{\sum (r_i - \mu)^2}{G}}
$$

$$
\sigma = \sqrt{\frac{(1.0 - 0.5)^2 + (0.0 - 0.5)^2 + (1.0 - 0.5)^2 + (0.0 - 0.5)^2}{4}}
= \sqrt{0.25} = 0.5
$$

### Step D: group relative advantage

We normalize each reward using the group statistics, adding a small stability term $\epsilon = 10^{-4}$:

$$
A_i = \frac{r_i - \mu}{\sigma + \epsilon}
$$

| Rollout | Raw reward | Calculation | Advantage |
| --- | ---: | --- | ---: |
| $o_{0,0}$ | 1.0 | $(1.0 - 0.5) / 0.5$ | $+1.0$ |
| $o_{0,1}$ | 0.0 | $(0.0 - 0.5) / 0.5$ | $-1.0$ |
| $o_{0,2}$ | 1.0 | $(1.0 - 0.5) / 0.5$ | $+1.0$ |
| $o_{0,3}$ | 0.0 | $(0.0 - 0.5) / 0.5$ | $-1.0$ |

### Interpretation

- Positive advantages boost token probabilities.
- Negative advantages suppress token probabilities.

## 4. Tensor Shapes and Token-Level Advantage Broadcasting

A critical detail is that $A_i$ is a single scalar per rollout, while an LLM generates a sequence of tokens of length $T_i$.

### Example

Rollout 0 has length 5 tokens:

```text
["2", "x", "=", "5", "<EOS>"]
```

If the scalar advantage is $+1.0$, it is broadcast across all generated tokens:

```text
Broadcasted advantage vector: [+1.0, +1.0, +1.0, +1.0, +1.0]
```

Every token produced in a winning trajectory receives a positive push, and every token in a losing trajectory receives a negative push.

## 5. Edge Cases in Group Advantage Normalization

In production, the advantage calculation must handle two important edge cases.

### Case 1: all rollouts get the same score

If all rewards are identical, such as $[0,0,0,0]$ or $[1,1,1,1]$, then:

- $\sigma = 0$
- $r_i - \mu = 0$
- $A_i = 0$

This produces zero gradient and avoids division-by-zero issues.

### Case 2: extreme outlier generation

With $G = 8$ and rewards $[0,0,0,0,0,0,0,1]$:

- the single correct answer receives a strong positive advantage
- the failed answers receive a mild negative advantage

This is useful because rare successes receive a strong gradient push while common failures receive only a mild penalty.
